# HDB Resale Price Prediction
**FYP-26-S1-13 | PropAI.sg**

Stacked generalisation model (XGBoost + CatBoost + LightGBM → HuberRegressor metalayer)
trained on HDB resale transactions fetched live from **data.gov.sg**.

### External data required (Google Drive)
- `data/policy_change.xlsx` — HDB policy change events
- `data/SORA_2017_2026.xlsx` — Monthly SORA 3-month compound rates
- `data/geocoded_addresses.csv` — Pre-geocoded block/street → lat/lon

## 1  Install & Import

In [ ]:
!pip install -q xgboost lightgbm catboost scikit-learn pandas numpy requests openpyxl

In [ ]:
import os, time, re, json, requests
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import HuberRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

## 2  Load HDB Resale Data — data.gov.sg API
Paginates through the full dataset (Jan 2017 → present). ~180k records.
Resource ID: `f1765b54-a209-4718-8d38-a39237f502b3`

In [ ]:
RESOURCE_ID = 'f1765b54-a209-4718-8d38-a39237f502b3'
BASE_URL    = 'https://data.gov.sg/api/action/datastore_search'

def fetch_hdb_data(resource_id=RESOURCE_ID, limit=10000):
    records, offset = [], 0
    print('Fetching HDB resale data from data.gov.sg...')
    while True:
        r = requests.get(BASE_URL,
                         params={'resource_id': resource_id, 'limit': limit, 'offset': offset},
                         timeout=60)
        r.raise_for_status()
        result  = r.json()['result']
        batch   = result['records']
        total   = result['total']
        records.extend(batch)
        offset  += len(batch)
        print(f'  {offset:,} / {total:,}', end='\r')
        if offset >= total:
            break
        time.sleep(0.3)
    print(f'\nDone — {len(records):,} records')
    return pd.DataFrame(records)

df = fetch_hdb_data()

## 3  Initial Exploration

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## 4  Deduplication

In [ ]:
key = ['month','town','flat_type','block','street_name',
       'storey_range','floor_area_sqm','flat_model',
       'lease_commence_date','remaining_lease','resale_price']
before = len(df)
df = df.drop_duplicates(subset=key, keep='first').reset_index(drop=True)
print(f'Dropped {before - len(df):,} duplicates — {len(df):,} rows remain')

## 5  Policy Change Data (Google Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

POLICY_PATH = '/content/drive/MyDrive/data/policy_change.xlsx'
policy = pd.read_excel(POLICY_PATH)
policy['effective_month'] = pd.to_datetime(policy['effective_month'])
policy = policy.sort_values('effective_month')

df['month'] = pd.to_datetime(df['month'].astype(str).str[:7] + '-01')
df = df.sort_values('month')

df = pd.merge_asof(
    df,
    policy[['effective_month','policy_name','category','direction','severity']]
        .sort_values('effective_month'),
    left_on='month', right_on='effective_month', direction='backward'
)

df['policy_impact'] = df['direction'] * df['severity']
df['months_since_policy_change'] = (
    (df['month'].dt.to_period('M') - df['effective_month'].dt.to_period('M'))
    .apply(lambda x: x.n if pd.notna(x) else None)
)

## 6  SORA Data (Google Drive)

In [ ]:
SORA_PATH = '/content/drive/MyDrive/data/SORA_2017_2026.xlsx'
sora = pd.read_excel(SORA_PATH)[['SORA Publication Date','Compound SORA - 3 month']].copy()
sora.columns = ['date','sora_3m']
sora['date'] = pd.to_datetime(sora['date'])
sora['month'] = sora['date'].dt.to_period('M').dt.to_timestamp()
sora_m = sora.groupby('month', as_index=False)['sora_3m'].mean().rename(columns={'sora_3m':'sora'})

df = df.merge(sora_m, on='month', how='left')

## 7  Feature Engineering

In [ ]:
df['year']    = df['month'].dt.year
df['quarter'] = df['month'].dt.quarter
df['time_idx'] = (
    (df['month'].dt.year - df['month'].dt.year.min()) * 12
    + df['month'].dt.month
)

# Storey range → numeric
sr = df['storey_range'].astype('string').str.upper().str.strip()
floors = sr.str.extract(r'(\d+)\s*TO\s*(\d+)')
df['storey_lo']  = pd.to_numeric(floors[0], errors='coerce')
df['storey_hi']  = pd.to_numeric(floors[1], errors='coerce')
df['storey_mid'] = (df['storey_lo'] + df['storey_hi']) / 2

# Remaining lease → years
rl = df['remaining_lease'].astype('string').str.upper()
y  = pd.to_numeric(rl.str.extract(r'(\d+)\s*YEAR')[0],  errors='coerce').fillna(0)
m  = pd.to_numeric(rl.str.extract(r'(\d+)\s*MONTH')[0], errors='coerce').fillna(0)
df['remaining_lease_months'] = (y * 12 + m).astype('Int64')
df['remaining_lease_years']  = (df['remaining_lease_months'] / 12).round(1)

# Flat age
df['lease_commence_date'] = pd.to_numeric(df['lease_commence_date'], errors='coerce').astype('Int64')
df['flat_age_years'] = df['year'] - df['lease_commence_date']

# Normalise strings
for c in ['town','flat_type','street_name','flat_model']:
    df[c] = df[c].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True).str.upper()
df['block'] = df['block'].astype('string').str.strip().str.upper()

# Target to numeric
df['resale_price'] = pd.to_numeric(df['resale_price'], errors='coerce')
df['floor_area_sqm'] = pd.to_numeric(df['floor_area_sqm'], errors='coerce')

df.head(3)

## 8  Geocoding (Google Drive — pre-computed)

In [ ]:
df['search_text'] = (
    df['block'].fillna('') + ' ' + df['street_name'].fillna('')
).str.strip().str.upper()

GEO_PATH = '/content/drive/MyDrive/data/geocoded_addresses.csv'
geo = pd.read_csv(GEO_PATH)
df  = df.merge(geo, on='search_text', how='left')
print(f'Missing geocodes: {df["lat"].isna().sum():,}')
df  = df.dropna(subset=['lat','lon'])
print(f'Rows after dropping missing geocodes: {len(df):,}')

## 9  Final Feature Set

In [ ]:
DROP_COLS = ['month','remaining_lease','storey_range','block',
             'lease_commence_date','category','remaining_lease_months',
             'storey_lo','storey_hi','effective_month','street_name','policy_name']
FINAL = df.drop(columns=[c for c in DROP_COLS if c in df.columns]).dropna(
    subset=['resale_price','floor_area_sqm','storey_mid','flat_age_years']
)
print(FINAL.shape)
FINAL.head(3)

## 10  Train / Test Split (time-based)

In [ ]:
categorical_cols = ['town','flat_type','flat_model']
numerical_cols   = [
    'floor_area_sqm','direction','severity','policy_impact',
    'months_since_policy_change','sora','year','quarter','time_idx',
    'storey_mid','flat_age_years','remaining_lease_years','lat','lon'
]
target_col = 'resale_price'

# Fill any remaining NaNs in numerical cols with median
for c in numerical_cols:
    FINAL[c] = pd.to_numeric(FINAL[c], errors='coerce')
    FINAL[c] = FINAL[c].fillna(FINAL[c].median())

train = FINAL[FINAL['year'] < 2025]
test  = FINAL[FINAL['year'] >= 2025]

X_train, y_train = train[categorical_cols + numerical_cols], train[target_col]
X_test,  y_test  = test[categorical_cols  + numerical_cols], test[target_col]

y_train_log = np.log(y_train)
y_test_log  = np.log(y_test)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

## 11  Base Model Pipelines

In [ ]:
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ('num', 'passthrough', numerical_cols)
])

xgb_pipeline = Pipeline([('pre', preprocessor), ('model', XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    objective='reg:squarederror', n_jobs=-1
))])

lgb_pipeline = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    n_jobs=-1, verbose=-1
))])

cat_pipeline = Pipeline([('pre', preprocessor), ('model', CatBoostRegressor(
    iterations=500, learning_rate=0.05, depth=6,
    random_seed=42, verbose=0
))])

## 12  Individual Model Evaluation

In [ ]:
def evaluate(name, pipeline, X_tr, X_te, y_tr, y_te):
    pipeline.fit(X_tr, np.log(y_tr))
    preds = np.exp(pipeline.predict(X_te))
    mae   = mean_absolute_error(y_te, preds)
    rmse  = np.sqrt(mean_squared_error(y_te, preds))
    r2    = r2_score(y_te, preds)
    mape  = mean_absolute_percentage_error(y_te, preds)
    print(f'{name:<20} MAE={mae:>10,.0f}  RMSE={rmse:>10,.0f}  R²={r2:.4f}  MAPE={mape*100:.2f}%')
    return preds

xgb_preds = evaluate('XGBoost',  xgb_pipeline, X_train, X_test, y_train, y_test)
lgb_preds = evaluate('LightGBM', lgb_pipeline, X_train, X_test, y_train, y_test)
cat_preds = evaluate('CatBoost', cat_pipeline, X_train, X_test, y_train, y_test)

## 13  Stacked Generalisation — XGB + LightGBM + CatBoost → HuberRegressor
Out-of-fold (OOF) predictions are used as meta-features so the metalayer
never sees the same rows it was trained on — preventing leakage.

In [ ]:
print('Generating OOF predictions (5-fold CV) — this may take a few minutes...')

# Re-instantiate clean pipelines for OOF (avoids re-using already-fitted ones)
xgb_oof_pipe = Pipeline([('pre', ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),('num','passthrough',numerical_cols)])), ('model', XGBRegressor(n_estimators=500,learning_rate=0.05,max_depth=6,subsample=0.8,colsample_bytree=0.8,random_state=42,objective='reg:squarederror',n_jobs=-1))])
lgb_oof_pipe = Pipeline([('pre', ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),('num','passthrough',numerical_cols)])), ('model', LGBMRegressor(n_estimators=500,learning_rate=0.05,num_leaves=63,subsample=0.8,colsample_bytree=0.8,random_state=42,n_jobs=-1,verbose=-1))])
cat_oof_pipe = Pipeline([('pre', ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),('num','passthrough',numerical_cols)])), ('model', CatBoostRegressor(iterations=500,learning_rate=0.05,depth=6,random_seed=42,verbose=0))])

xgb_oof = cross_val_predict(xgb_oof_pipe, X_train, y_train_log, cv=5, n_jobs=-1)
print('XGBoost OOF done')
lgb_oof = cross_val_predict(lgb_oof_pipe, X_train, y_train_log, cv=5, n_jobs=-1)
print('LightGBM OOF done')
cat_oof = cross_val_predict(cat_oof_pipe, X_train, y_train_log, cv=5)
print('CatBoost OOF done')

In [ ]:
# Build meta-features and train HuberRegressor metalayer
meta_X_train = np.column_stack([xgb_oof, lgb_oof, cat_oof])

meta_model = HuberRegressor(epsilon=1.35, max_iter=300)
meta_model.fit(meta_X_train, y_train_log)
print('Metalayer weights (XGB, LGB, CAT):', meta_model.coef_.round(4))

In [ ]:
# Test-set predictions: get log-space preds from each base model, then stack
xgb_test_log = xgb_pipeline.predict(X_test)   # already fitted in section 12
lgb_test_log = lgb_pipeline.predict(X_test)
cat_test_log = cat_pipeline.predict(X_test)

meta_X_test   = np.column_stack([xgb_test_log, lgb_test_log, cat_test_log])
stacked_log   = meta_model.predict(meta_X_test)
stacked_preds = np.exp(stacked_log)

s_mae  = mean_absolute_error(y_test, stacked_preds)
s_rmse = np.sqrt(mean_squared_error(y_test, stacked_preds))
s_r2   = r2_score(y_test, stacked_preds)
s_mape = mean_absolute_percentage_error(y_test, stacked_preds)
print(f'Stacked (HuberMeta)   MAE={s_mae:>10,.0f}  RMSE={s_rmse:>10,.0f}  R²={s_r2:.4f}  MAPE={s_mape*100:.2f}%')

## 14  Evaluation Summary

In [ ]:
results = pd.DataFrame({
    'Model':['XGBoost','LightGBM','CatBoost','Stacked (Huber)'],
    'MAE':  [mean_absolute_error(y_test, xgb_preds),
             mean_absolute_error(y_test, lgb_preds),
             mean_absolute_error(y_test, cat_preds),
             s_mae],
    'RMSE': [np.sqrt(mean_squared_error(y_test, p)) for p in [xgb_preds,lgb_preds,cat_preds]] + [s_rmse],
    'R2':   [r2_score(y_test, p) for p in [xgb_preds,lgb_preds,cat_preds]] + [s_r2],
    'MAPE': [mean_absolute_percentage_error(y_test, p)*100 for p in [xgb_preds,lgb_preds,cat_preds]] + [s_mape*100]
})
results = results.sort_values('MAE').reset_index(drop=True)
print(results.to_string(index=False))

## 15  Sample Predictions

In [ ]:
sample = X_test.iloc[:10].copy()
actual = y_test.iloc[:10].values

s_xgb = np.exp(xgb_pipeline.predict(sample))
s_lgb = np.exp(lgb_pipeline.predict(sample))
s_cat = np.exp(cat_pipeline.predict(sample))
s_stk = np.exp(meta_model.predict(np.column_stack([
    xgb_pipeline.predict(sample),
    lgb_pipeline.predict(sample),
    cat_pipeline.predict(sample)
])))

out = pd.DataFrame({
    'Actual':          actual,
    'XGBoost':         s_xgb,
    'LightGBM':        s_lgb,
    'CatBoost':        s_cat,
    'Stacked (Huber)': s_stk,
})
out['Best Error $'] = out.apply(
    lambda r: min(abs(r['Actual']-r[m]) for m in ['XGBoost','LightGBM','CatBoost','Stacked (Huber)']), axis=1
)
out['Stacked Error %'] = (abs(out['Actual'] - out['Stacked (Huber)']) / out['Actual'] * 100).map('{:.2f}%'.format)
for c in ['Actual','XGBoost','LightGBM','CatBoost','Stacked (Huber)','Best Error $']:
    out[c] = out[c].map('${:,.0f}'.format)
out